# IV Ratio Ranking: Predicting Calendar Spread Profitability

This notebook explores whether **pre-earnings IV ratios** can predict calendar spread profitability.

## Hypothesis

Higher IV ratio (short-term IV / long-term IV) at entry → better P&L

**Rationale**:
- High ratio means short-term options are "expensive" relative to long-term
- After earnings, short-term IV drops more (IV crush)
- Spread value increases → profit

## Methodology

1. Backtest calendar spreads across all earnings events (reuse notebook 04 logic)
2. Calculate IV for each leg at entry using Black-Scholes
3. Compute IV ratio: $\frac{IV_{short}}{IV_{long}}$
4. Analyze correlation between IV ratio and P&L
5. Rank opportunities by IV ratio and compare performance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

from dlt_ibapi.repositories import (
    EarningsCalendarReader,
    OptionBarsReader,
    EquityBarsReader
)
from dlt_ibapi.strategies import (
    filter_tradable_earnings,
    run_batch_calendar_spread_backtest,
    calculate_strategy_stats,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Setup complete")

## 1. Load and Filter Earnings Events

In [ ]:
# Load earnings data
earnings_reader = EarningsCalendarReader(database_path='../data_delta', dataset_name='earnings')
all_earnings = earnings_reader.get_upcoming_earnings(
    days_ahead=365,
    from_date=date(2025, 1, 1)
)

print(f"Total earnings events: {len(all_earnings)}")

# Filter to tradable events
option_reader = OptionBarsReader(database_path='../data_delta', dataset_name='options')
tradable_earnings = filter_tradable_earnings(all_earnings, option_reader)

print(f"Tradable earnings (with option data): {len(tradable_earnings)}")
print(f"Symbols: {len(tradable_earnings['symbol'].unique())} unique")

## 2. Run Backtest with IV Calculation

This time we enable `calculate_iv=True` to compute implied volatility for each leg.

In [ ]:
# Initialize readers
equity_reader = EquityBarsReader(database_path='../data_delta', dataset_name='stocks')

# Run batch backtest WITH IV calculation
results_df = run_batch_calendar_spread_backtest(
    earnings_df=tradable_earnings,
    option_reader=option_reader,
    equity_reader=equity_reader,
    option_type='C',
    bar_size='1 hour',
    progress=True,
    calculate_iv=True  # NEW: Calculate IV metrics
)

print(f"\n{'='*80}")
print(f"BACKTEST COMPLETE")
print(f"{'='*80}")
print(f"Successful backtests: {len(results_df)}")
print(f"Success rate: {len(results_df) / len(tradable_earnings) * 100:.1f}%")

## 3. Filter to Valid IV Data

Only analyze trades where we successfully calculated IV for both legs.

In [ ]:
# Filter to trades with valid IV data
valid_iv_df = results_df[
    results_df['iv_short_entry'].notna() & 
    results_df['iv_long_entry'].notna() &
    results_df['iv_ratio_entry'].notna()
].copy()

print(f"Trades with valid IV: {len(valid_iv_df)} / {len(results_df)}")

if len(valid_iv_df) > 0:
    print(f"\nIV Ratio Statistics:")
    print(f"  Mean: {valid_iv_df['iv_ratio_entry'].mean():.3f}")
    print(f"  Median: {valid_iv_df['iv_ratio_entry'].median():.3f}")
    print(f"  Min: {valid_iv_df['iv_ratio_entry'].min():.3f}")
    print(f"  Max: {valid_iv_df['iv_ratio_entry'].max():.3f}")
    print(f"\nSample trades:")
    print(valid_iv_df[['symbol', 'iv_short_entry', 'iv_long_entry', 'iv_ratio_entry', 'pnl_per_contract']].head(10))
else:
    print("⚠️ No trades with valid IV data")

## 4. Correlation Analysis: IV Ratio vs P&L

Test the hypothesis: Does higher IV ratio predict better P&L?

In [ ]:
if len(valid_iv_df) > 0:
    correlation = valid_iv_df['iv_ratio_entry'].corr(valid_iv_df['pnl_per_contract'])
    
    print("="*80)
    print("CORRELATION ANALYSIS: IV Ratio vs P&L")
    print("="*80)
    print(f"Pearson correlation: {correlation:.3f}")
    
    if correlation > 0.3:
        print("✅ Strong positive correlation: Higher IV ratio → Better P&L")
    elif correlation > 0.1:
        print("⚠️ Weak positive correlation")
    elif correlation < -0.1:
        print("❌ Negative correlation: Higher IV ratio → Worse P&L")
    else:
        print("❓ No significant correlation")
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(valid_iv_df['iv_ratio_entry'], valid_iv_df['pnl_per_contract'], alpha=0.6)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1, label='Breakeven')
    
    # Add trend line
    z = pd.np.polyfit(valid_iv_df['iv_ratio_entry'], valid_iv_df['pnl_per_contract'], 1)
    p = pd.np.poly1d(z)
    ax.plot(valid_iv_df['iv_ratio_entry'], p(valid_iv_df['iv_ratio_entry']), 
            "r-", linewidth=2, label=f'Trend (r={correlation:.3f})')
    
    ax.set_xlabel('IV Ratio at Entry (Short IV / Long IV)', fontsize=12, fontweight='bold')
    ax.set_ylabel('P&L per Contract ($)', fontsize=12, fontweight='bold')
    ax.set_title('Calendar Spread P&L vs Pre-Earnings IV Ratio', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data for correlation analysis")

## 5. Rank Opportunities by IV Ratio

Divide trades into quartiles by IV ratio and compare performance.

In [ ]:
if len(valid_iv_df) > 0:
    # Create quartiles
    valid_iv_df['iv_ratio_quartile'] = pd.qcut(
        valid_iv_df['iv_ratio_entry'], 
        q=4, 
        labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
    )
    
    # Analyze by quartile
    quartile_stats = valid_iv_df.groupby('iv_ratio_quartile').agg({
        'pnl_per_contract': ['count', 'mean', 'median', 'std'],
        'iv_ratio_entry': ['min', 'max'],
    }).round(2)
    
    quartile_stats.columns = ['Count', 'Mean P&L', 'Median P&L', 'Std Dev', 'IV Ratio Min', 'IV Ratio Max']
    
    print("="*80)
    print("PERFORMANCE BY IV RATIO QUARTILE")
    print("="*80)
    print(quartile_stats)
    
    # Win rate by quartile
    win_rates = valid_iv_df.groupby('iv_ratio_quartile').apply(
        lambda x: (x['pnl_per_contract'] > 0).sum() / len(x) * 100
    )
    print(f"\nWin Rate by Quartile:")
    for q, wr in win_rates.items():
        print(f"  {q}: {wr:.1f}%")
    
    # Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Mean P&L by quartile
    ax1 = axes[0]
    quartile_stats['Mean P&L'].plot(kind='bar', ax=ax1, color='steelblue', alpha=0.7)
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax1.set_xlabel('IV Ratio Quartile', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Mean P&L per Contract ($)', fontsize=11, fontweight='bold')
    ax1.set_title('Average P&L by IV Ratio Quartile', fontsize=12, fontweight='bold')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Win rate by quartile
    ax2 = axes[1]
    win_rates.plot(kind='bar', ax=ax2, color='green', alpha=0.7)
    ax2.set_xlabel('IV Ratio Quartile', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Win Rate (%)', fontsize=11, fontweight='bold')
    ax2.set_title('Win Rate by IV Ratio Quartile', fontsize=12, fontweight='bold')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data for quartile analysis")

## 6. Compare Top vs Bottom Quartile

Direct comparison: Best opportunities (highest IV ratio) vs worst (lowest IV ratio).

In [ ]:
if len(valid_iv_df) >= 8:  # Need at least 8 trades for quartiles
    top_quartile = valid_iv_df[valid_iv_df['iv_ratio_quartile'] == 'Q4 (High)']
    bottom_quartile = valid_iv_df[valid_iv_df['iv_ratio_quartile'] == 'Q1 (Low)']
    
    print("="*80)
    print("TOP QUARTILE (Highest IV Ratio) vs BOTTOM QUARTILE (Lowest IV Ratio)")
    print("="*80)
    
    print(f"\nTOP QUARTILE (Q4):")
    print(f"  Count: {len(top_quartile)}")
    print(f"  IV Ratio Range: {top_quartile['iv_ratio_entry'].min():.3f} - {top_quartile['iv_ratio_entry'].max():.3f}")
    print(f"  Mean P&L: ${top_quartile['pnl_per_contract'].mean():.2f}")
    print(f"  Win Rate: {(top_quartile['pnl_per_contract'] > 0).sum() / len(top_quartile) * 100:.1f}%")
    
    print(f"\nBOTTOM QUARTILE (Q1):")
    print(f"  Count: {len(bottom_quartile)}")
    print(f"  IV Ratio Range: {bottom_quartile['iv_ratio_entry'].min():.3f} - {bottom_quartile['iv_ratio_entry'].max():.3f}")
    print(f"  Mean P&L: ${bottom_quartile['pnl_per_contract'].mean():.2f}")
    print(f"  Win Rate: {(bottom_quartile['pnl_per_contract'] > 0).sum() / len(bottom_quartile) * 100:.1f}%")
    
    diff_pnl = top_quartile['pnl_per_contract'].mean() - bottom_quartile['pnl_per_contract'].mean()
    print(f"\n💰 DIFFERENCE: ${diff_pnl:.2f} per contract")
    
    if diff_pnl > 50:
        print("✅ STRONG SIGNAL: High IV ratio significantly outperforms low IV ratio")
    elif diff_pnl > 20:
        print("✅ MODERATE SIGNAL: High IV ratio tends to outperform")
    elif diff_pnl < -20:
        print("❌ INVERSE SIGNAL: Low IV ratio performs better (hypothesis rejected)")
    else:
        print("⚠️ WEAK SIGNAL: No significant difference")
else:
    print("⚠️ Insufficient data for quartile comparison (need at least 8 trades)")

## 7. Inspect Best and Worst Trades

Look at individual trades ranked by IV ratio.

In [ ]:
if len(valid_iv_df) > 0:
    # Sort by IV ratio
    sorted_df = valid_iv_df.sort_values('iv_ratio_entry', ascending=False)
    
    print("="*80)
    print("TOP 10 OPPORTUNITIES (Highest IV Ratio)")
    print("="*80)
    print(sorted_df[[
        'symbol', 'iv_ratio_entry', 'iv_short_entry', 'iv_long_entry', 
        'entry_cost_per_contract', 'pnl_per_contract', 'pnl_pct'
    ]].head(10).to_string(index=False))
    
    print(f"\n{'='*80}")
    print("BOTTOM 10 OPPORTUNITIES (Lowest IV Ratio)")
    print("="*80)
    print(sorted_df[[
        'symbol', 'iv_ratio_entry', 'iv_short_entry', 'iv_long_entry', 
        'entry_cost_per_contract', 'pnl_per_contract', 'pnl_pct'
    ]].tail(10).to_string(index=False))
else:
    print("⚠️ No trades to inspect")

## 8. Save Results

In [ ]:
if len(valid_iv_df) > 0:
    valid_iv_df.to_csv('iv_ratio_ranking_results.csv', index=False)
    print(f"✅ Results saved to iv_ratio_ranking_results.csv ({len(valid_iv_df)} trades)")
    
    # Summary stats
    print(f"\n{'='*80}")
    print("SUMMARY")
    print("="*80)
    print(f"Total trades analyzed: {len(valid_iv_df)}")
    print(f"Mean IV ratio: {valid_iv_df['iv_ratio_entry'].mean():.3f}")
    print(f"Mean P&L: ${valid_iv_df['pnl_per_contract'].mean():.2f}")
    print(f"Correlation (IV ratio vs P&L): {valid_iv_df['iv_ratio_entry'].corr(valid_iv_df['pnl_per_contract']):.3f}")
else:
    print("⚠️ No results to save")

## 9. Conclusions

**Key Questions**:
1. Is there a correlation between pre-earnings IV ratio and P&L?
2. Does ranking by IV ratio help select better opportunities?
3. What threshold IV ratio should we use for trade selection?

**Next Steps**:
- If correlation is strong: Use IV ratio as a filter in live trading
- If correlation is weak: Explore other signals (absolute IV level, DTE, moneyness, etc.)
- Combine with other notebooks (03 for single-trade analysis, 04 for batch backtest, 05 for vol surfaces)